In [1]:
import random

In [ ]:
class CBMInstanceGenerator:
    def __init__(self,
                 lines: int,
                 cols: int,
                 min_density: float,
                 max_density: float,
                 output_file: str = "instance.txt",
                 seed: int | None = None):
        """
        lines        -> number of rows
        cols         -> number of columns
        min_density  -> minimum % of 1s per column (0–1)
        max_density  -> maximum % of 1s per column (0–1)
        output_file  -> path to save instance
        seed         -> optional RNG seed for reproducibility
        """

        if not (0 <= min_density <= max_density <= 1):
            raise ValueError("Density must satisfy 0 <= min <= max <= 1")

        self.lines = lines
        self.cols = cols
        self.min_density = min_density
        self.max_density = max_density
        self.output_file = output_file

        self.random = random.Random(seed)

        # matrix stored internally only if user wants to inspect it
        self.matrix = None

    # ---------- generation logic ----------
    def _generate_matrix(self):
        matrix = [[0 for _ in range(self.cols)] for _ in range(self.lines)]

        for col in range(self.cols):
            min_ones = int(self.min_density * self.lines)
            max_ones = int(self.max_density * self.lines)

            if max_ones == 0 and self.max_density > 0:
                max_ones = 1
            if min_ones == 0 and self.min_density > 0:
                min_ones = 1

            k = self.random.randint(min_ones, max_ones)
            rows = self.random.sample(range(self.lines), k)

            for r in rows:
                matrix[r][col] = 1

        return matrix

    def generate(self, store_matrix: bool = False):
        matrix = self._generate_matrix()

        if store_matrix:
            self.matrix = matrix

        self._write_file(matrix)

    def _write_file(self, matrix):
        with open(self.output_file, "w") as f:
            f.write(f"{self.lines} {self.cols}\n")

            for row in matrix:
                ones = [str(i + 1) for i, v in enumerate(row) if v == 1]
                f.write(str(len(ones)))
                if ones:
                    f.write(" " + " ".join(ones))
                f.write("\n")

    def stats(self):
        if self.matrix is None:
            print("Matrix not stored. Run generate(store_matrix=True).")
            return

        col_counts = [sum(self.matrix[r][c] for r in range(self.lines))
                      for c in range(self.cols)]

        print("Column density stats:")
        print("min:", min(col_counts) / self.lines)
        print("max:", max(col_counts) / self.lines)
        print("avg:", sum(col_counts) / (self.cols * self.lines))

In [ ]:
gen = CBMInstanceGenerator(
    lines=100,
    cols=50,
    min_density=0.2,
    max_density=0.4,
    output_file="instance.txt",
    seed=42
)

gen.generate(store_matrix=True)
gen.stats()